In [0]:
%pip install -q mlforecast lightgbm utilsforecast xgboost statsforecast
!pip install unidecode

In [0]:
from utilsforecast.evaluation import evaluate
from utilsforecast.losses import mae, rmse, mape, mase, bias, cfe, smape
from utilsforecast.plotting import plot_series

from datetime import datetime, date, timedelta

from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.functions import col, count, sum, lit, when, regexp_extract

from mlforecast import MLForecast
from mlforecast.target_transforms import Differences, LocalBoxCox
from mlforecast.lag_transforms import RollingMean, ExpandingMean
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
from sklearn.linear_model import LinearRegression
import pandas as pd

from pyspark.sql import SparkSession, DataFrame
from pyspark.sql.types import StringType, NullType, StructType

import matplotlib.pyplot as plt

In [0]:
def get_table_schema(table_name: str, spark: SparkSession) -> StructType:
        return spark.table(table_name).schema
   
 
def apply_schema_from_table(df, table_name: str,spark: SparkSession, fill_missing_with_zero=True):
    target_schema = get_table_schema(table_name,spark)
   
    cols_expr = []
   
    for field in target_schema:
        col_name = field.name
        col_type = field.dataType
       
        if col_name in df.columns:
            # cast para o tipo correto
            col_expr = F.col(col_name).cast(col_type)
        else:
            # cria coluna faltante
            if fill_missing_with_zero and col_type.simpleString() in ["bigint", "int", "double", "float"]:
                col_expr = F.lit(0).cast(col_type)
            else:
                col_expr = F.lit(None).cast(col_type)
       
        cols_expr.append(col_expr.alias(col_name))
   
    return df.select(*cols_expr)

In [0]:
dbutils.widgets.text("catalog", "", "Catalog")
catalog = dbutils.widgets.get("catalog")
if not catalog:
    catalog = "planejamento_integrado_dev"
print(catalog)

today_str = (datetime.utcnow() - timedelta(hours=3)).strftime("%Y-%m-%d")
#today_str = '2025-01-31'
today_date = datetime.strptime(today_str, "%Y-%m-%d").date()
print("today_str :", today_str)

yesterday_date = today_date - timedelta(days=1)
yesterday_str = yesterday_date.strftime("%Y-%m-%d")
print("yesterday_str:", yesterday_str)


# Regra DFU

In [0]:
regra_dfu_cirurgia = (
    # ROBÓTICA
    F.when(F.col('fl_robotica') == 1, "Robótica")

    # INFANTIL por idade
    .when(
        F.col('nr_anos') <= 14, "Infantil"
    )

    # INFANTIL pela especialidade
    .when(F.col("ds_especialidade_cirurgiao").isin([
        "Cirurgia Pediátrica",
        "Otorrinolaringologia Pediátrica",
        "Pediatria",
        "Medicina Intensiva Pediátrica",
        "Neonatologia",
        "Odontopediatria",
        "Cardiologia Pediatrica",
        "Endocrinologia Pediátrica"
    ]), "Infantil")

    # Ortopedia
    .when(
        F.col("ds_especialidade_cirurgiao") == 'Ortopedia e Traumatologia',
        "Ortopedia"
    )

    # GINECOLOGIA E OBSTETRÍCIA
    .when(F.col("ds_especialidade_cirurgiao").isin([
        "Ginecologia e Obstetrícia"
    ]), "Ginecologia e Obstetrícia")
    
    # CIRURGIA GERAL (Abdomem, Onco, Urologia)
    .when(F.col("ds_especialidade_cirurgiao").isin([
        "Urologia", # Solicitado explicitamente
        "Cirurgia Geral",
        "Radiologia",
        
        # Oncológicas
        "Cirurgia Oncológica",
        "Cancerologia Cirúrgica",
        "Mastologia",

    ]), "Cirurgia Geral")

    # OUTROS
    .otherwise("Outros")
)

regra_dfu_agendamento = (
    # ROBÓTICA
    F.when(F.col('fl_robotica') == 1, "Robótica")

    # INFANTIL por idade
    .when(
        F.col('nr_anos') <= 14, "Infantil"
    )

    # INFANTIL pela especialidade
    .when(F.col("ds_especialidade").isin([
        "Cirurgia Pediátrica",
        "Otorrinolaringologia Pediátrica",
        "Pediatria",
        "Medicina Intensiva Pediátrica",
        "Neonatologia",
        "Odontopediatria",
        "Cardiologia Pediatrica",
        "Endocrinologia Pediátrica"
    ]), "Infantil")

    # Ortopedia
    .when(
        F.col("ds_especialidade") == 'Ortopedia e Traumatologia',
        "Ortopedia"
    )

    # GINECOLOGIA E OBSTETRÍCIA
    .when(F.col("ds_especialidade").isin([
        "Ginecologia e Obstetrícia"
    ]), "Ginecologia e Obstetrícia")
    
    # CIRURGIA GERAL (Abdomem, Onco, Urologia)
    .when(F.col("ds_especialidade").isin([
        "Urologia", # Solicitado explicitamente
        "Cirurgia Geral",
        "Radiologia",
        
        # Oncológicas
        "Cirurgia Oncológica",
        "Cancerologia Cirúrgica",
        "Mastologia",

    ]), "Cirurgia Geral")

    # OUTROS
    .otherwise("Outros")
)

In [0]:
# === Observabilidade: setup ===
import os
import sys
import importlib

current_dir = os.getcwd()
parent_dir = os.path.dirname(current_dir)
if parent_dir not in sys.path:
    sys.path.append(parent_dir)

import utils
importlib.reload(utils)
from utils import start_pipeline_log, end_pipeline_log

# timestamp usado como dt_referencia no log
timestamp = today_str.replace('-', '') + '_' + datetime.now().strftime('%H%M%S')


def get_databricks_context():
    job_id = None
    task_id = None
    notebook_path = None

    try:
        ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()

        job_id = ctx.jobId().get() if ctx.jobId().isDefined() else None
        task_id = ctx.currentRunId().get() if ctx.currentRunId().isDefined() else None
        notebook_path = ctx.notebookPath().get() if ctx.notebookPath().isDefined() else None

    except Exception as e:
        print(f"Could not get context: {e}")

    return job_id, task_id, notebook_path


job_id, task_id, notebook_path = get_databricks_context()


In [0]:
run_id = start_pipeline_log(
    spark=spark,
    pipeline_name="pi_previsao_cirurgias",
    step_name="previsao_eletivas.load_cirurgias_e_agendamentos",
    job_id=job_id,
    task_id=task_id,
    notebook_path=notebook_path,
    dt_referencia=timestamp,
    source_table="bi_dev.gold.hc_cirurgia + bi_dev.gold.hc_agenda_cirurgica",
    target_table="cirurgias_df + agendamentos_df"
)

try:

    cirurgias = spark.sql("""
    WITH cirurgias_robo AS (
        -- Seleciona apenas os registros que indicam cirurgia robótica
        SELECT 
            hub,
            nr_atendimento,
            cd_medico_cirurgiao,
            cd_pessoa_fisica,
            CAST(dt_inicio_real AS DATE) AS data_cirurgia,
            ds_sala
        FROM bi_dev.gold.hc_cirurgia
        WHERE lower(ds_procedimento) like '%robótica%'
    ),

    idades AS (
        select distinct hub, cd_pessoa_fisica, nr_anos from bi_dev.gold.hc_pessoa_fisica
    )

    SELECT 
        base.*,
        idade.nr_anos,
        -- Cria a flag: Se encontrou match na CTE de robôs, é 1, senão 0
        CASE 
            WHEN robo.nr_atendimento IS NOT NULL THEN 1 
            ELSE 0 
        END AS fl_robotica
    FROM bi_dev.gold.hc_cirurgia AS base
    LEFT JOIN cirurgias_robo AS robo
        ON base.nr_atendimento = robo.nr_atendimento
        AND base.hub = robo.hub
        AND base.cd_medico_cirurgiao = robo.cd_medico_cirurgiao
        AND base.cd_pessoa_fisica = robo.cd_pessoa_fisica
        AND CAST(base.dt_inicio_real AS DATE) = robo.data_cirurgia
        AND base.ds_sala = robo.ds_sala
    LEFT JOIN idades AS idade
        ON base.cd_pessoa_fisica = idade.cd_pessoa_fisica
        AND base.hub = idade.hub
    WHERE 
        -- Exclui a linha duplicada que serve apenas de marcador
        lower(base.ds_procedimento) not like '%robótica%'
        and base.hub = 'Campinas';
    """).withColumn('dfu', regra_dfu_cirurgia)

    agendamentos = spark.sql("""

    WITH agendamentos_robo AS (
        -- Seleciona apenas os registros de agendamento do robô
        SELECT 
            hub,
            cd_medico,
            cd_pessoa_fisica,
            CAST(dt_agenda AS DATE) AS data_agenda,
            ds_agenda
        FROM bi_dev.gold.hc_agenda_cirurgica
        WHERE lower(ds_procedimento) like '%robótica%'
    ),

    idades AS (
        select distinct hub, cd_pessoa_fisica, nr_anos from bi_dev.gold.hc_pessoa_fisica
    )

    SELECT 
        base.*,
        idade.nr_anos,
        -- Cria a flag identificando se existe vínculo robótico,
        CASE 
            WHEN robo.cd_pessoa_fisica IS NOT NULL THEN 1 
            ELSE 0 
        END AS fl_robotica
    FROM bi_dev.gold.hc_agenda_cirurgica AS base
    LEFT JOIN agendamentos_robo AS robo
        ON base.cd_medico = robo.cd_medico
        AND base.hub = robo.hub
        AND base.cd_pessoa_fisica = robo.cd_pessoa_fisica
        AND CAST(base.dt_agenda AS DATE) = robo.data_agenda
        AND base.ds_agenda = robo.ds_agenda
    LEFT JOIN idades AS idade
        ON base.cd_pessoa_fisica = idade.cd_pessoa_fisica
        AND base.hub = idade.hub
    WHERE 
        -- Exclui a linha duplicada de marcador robótico
        lower(base.ds_procedimento) not like '%robótica%'
        and base.hub = 'Campinas';
    """).withColumn('dfu', regra_dfu_agendamento)

    rows_written = cirurgias.count() + agendamentos.count()
    end_pipeline_log(
        spark,
        run_id,
        status="SUCCESS",
        rows_written=rows_written,
    )

except Exception as e:
    safe_error = str(e).replace("'", "''")
    end_pipeline_log(
        spark,
        run_id,
        status="FAILED",
        error_message=safe_error
    )
    raise


In [0]:
run_id = start_pipeline_log(
    spark=spark,
    pipeline_name="pi_previsao_cirurgias",
    step_name="previsao_eletivas.load_feriados",
    job_id=job_id,
    task_id=task_id,
    notebook_path=notebook_path,
    dt_referencia=timestamp,
    source_table="planejamento_integrado.gold.calendario",
    target_table="feriados_df"
)

try:
    feriados = spark.table('planejamento_integrado.gold.calendario')\
        .where("cidade = 'Campinas'")\
        .select(F.col('data').alias('data_cirurgia'), F.when(F.col("feriado").isNotNull(), 1).otherwise(0).alias('feriado'))\
        .orderBy('data_cirurgia')

    feriados.display()

    rows_written = feriados.count()
    end_pipeline_log(
        spark,
        run_id,
        status="SUCCESS",
        rows_written=rows_written,
    )

except Exception as e:
    safe_error = str(e).replace("'", "''")
    end_pipeline_log(
        spark,
        run_id,
        status="FAILED",
        error_message=safe_error
    )
    raise


In [0]:
run_id = start_pipeline_log(
    spark=spark,
    pipeline_name="pi_previsao_cirurgias",
    step_name="previsao_eletivas.build_serie_historica",
    job_id=job_id,
    task_id=task_id,
    notebook_path=notebook_path,
    dt_referencia=timestamp,
    source_table="cirurgias + agendamentos + feriados",
    target_table="resultado_final (serie histórica)"
)

try:

    # ==============================================================================
    # 1. PREPARAR BASE DE AGENDAMENTOS (hc_agenda_cirurgica)
    # ==============================================================================

    agenda_df = agendamentos \
        .where(col('hub') == "Campinas") \
        .where(~F.col('ds_agenda').like('%VIRTUAL%')) \
        .where(col('ds_carater').like('%Eletiva%')) \
        .where(~col('ds_agenda').isin(['Ressonância', 'Tomografia'])) \
        .where(col('ds_estabelecimento') == 'Hospital Vera Cruz S/a')

    data_extraida_str = regexp_extract(col("ds_motivo_cancelamento"), r"em (\d{2}/\d{2}/\d{4})", 1)
    data_extraida_date = F.to_date(data_extraida_str, 'dd/MM/yyyy')

    data_cancelamento_final = F.coalesce(
        F.to_date(col('dt_cancelamento')),
        data_extraida_date
    )

    agenda_com_cancelamento_unificado = agenda_df.withColumn(
        "data_cancelamento_final", data_cancelamento_final
    ).withColumn("data_cancelamento_final", 
                 F.when(col('cd_motivo_cancelamento').isNull(), F.lit(None))
                 .otherwise(F.col('data_cancelamento_final')))

    base_agendamentos = agenda_com_cancelamento_unificado.select(
        F.to_date('dt_agenda').alias('data_cirurgia'),
        F.col('dfu'), 
        F.col('nr_cirurgia').cast('string').alias('nr_cirurgia'),
        F.datediff(F.to_date('dt_agenda'), F.to_date('dt_agendamento')).alias('antecedencia_agendamento'),
        F.datediff(F.to_date('dt_agenda'), col('data_cancelamento_final')).alias('antecedencia_cancelamento')
    ).where(
        col('data_cirurgia').isNotNull() &
        col('antecedencia_agendamento').isNotNull() &
        (col('antecedencia_agendamento') >= 0)
    )

    # ==============================================================================
    # 2. PREPARAR BASE DE CIRURGIAS REALIZADAS (hc_cirurgia)
    # ==============================================================================

    cirurgias_raw = cirurgias.where(
        (col('hub') == "Campinas") &
        (col('cd_estabelecimento') == 2) &
        (col('ds_status_cirurgia') == "Realizada") &
        (col('ds_carater').like('%Eletiva%'))
    )

    cirurgias_df = cirurgias_raw

    totais_realizadas_df = cirurgias_df.groupBy(
        F.to_date('dt_inicio_real').alias("data_cirurgia"),
        col("dfu")
    ).agg(
        count("*").alias("total_cirurgias_realizadas_dia")
    ).where(col('data_cirurgia').isNotNull())

    dfu_validos = [row['dfu'] for row in agendamentos.select('dfu').distinct().collect()]

    # ==============================================================================
    # 3. CRIAR O GABARITO (SCAFFOLD) COM DFUs
    # ==============================================================================

    # 3.1 Datas
    min_max_df = spark.range(1).select(
        F.lit('2023-01-01').cast('date').alias('min_data'),
        F.date_sub(F.current_date(), 1).alias('max_data')
    )

    todas_datas = min_max_df.select(
        F.explode(
            F.sequence(col('min_data'), col('max_data'), F.expr('interval 1 day'))
        ).alias('data_cirurgia')
    )

    # 3.2 Dias de Antecedência
    dias_gabarito = spark.range(1, 51).withColumnRenamed("id", "dias_antecedencia")

    # 3.3 dfu (Criamos um DF fixo com os DFUs desejados para garantir que apareçam sempre)
    lista_dfu = [(s,) for s in dfu_validos]
    df_dfu = spark.createDataFrame(lista_dfu, ["dfu"])

    # 3.4 Cruzamento Total (Datas x Dias x DFU)
    # Isso garante a linha mesmo se volume for 0
    scaffold_df = todas_datas.crossJoin(dias_gabarito).crossJoin(df_dfu)


    # ==============================================================================
    # 4. JUNTAR GABARITO E APLICAR LÓGICA
    # ==============================================================================

    # Join agora utiliza 'data_cirurgia' E 'dfu'
    joined_df = scaffold_df.join(base_agendamentos, ["data_cirurgia", "dfu"], "left")

    condicao_ativo = (
        (col("dias_antecedencia") <= col("antecedencia_agendamento")) &
        (
            (col("antecedencia_cancelamento").isNull()) |
            (col("dias_antecedencia") > col("antecedencia_cancelamento"))
        )
    )

    # Agrupamento inclui 'dfu'
    contagem_df = joined_df.groupBy("dfu", "data_cirurgia", "dias_antecedencia") \
        .agg(
            count(when(condicao_ativo, 1)).alias("qtd_acumulada_ate_antecedencia")
        )


    # ==============================================================================
    # 5. MONTAGEM FINAL
    # ==============================================================================

    # Join com as realizadas também por data e dfu
    resultado_final = contagem_df.join(
        totais_realizadas_df,
        ["data_cirurgia", "dfu"],
        "left"
    ).fillna(
        0, subset=["total_cirurgias_realizadas_dia"]
    ).select(
        col("dfu"),
        col("data_cirurgia"),
        col("dias_antecedencia"),
        col("total_cirurgias_realizadas_dia").cast("long"),
        col("qtd_acumulada_ate_antecedencia").cast("long").alias('agendamentos')
    ).join(
        feriados,
        on=['data_cirurgia'],
        how='left'
    ).withColumn('periodo_baixa', 
        F.when((F.month('data_cirurgia') == 12) & (F.dayofmonth('data_cirurgia') >= 22), 1)
        .when((F.month('data_cirurgia') == 1) & (F.dayofmonth('data_cirurgia') <= 5), 1)
        .otherwise(0)
    )

    # Exibir
    resultado_final \
        .orderBy(col('dfu'), F.desc('data_cirurgia'), 'dias_antecedencia') \
        .display()

    rows_written = resultado_final.count()
    end_pipeline_log(
        spark,
        run_id,
        status="SUCCESS",
        rows_written=rows_written,
    )

except Exception as e:
    safe_error = str(e).replace("'", "''")
    end_pipeline_log(
        spark,
        run_id,
        status="FAILED",
        error_message=safe_error
    )
    raise


In [0]:
dfus = [row['dfu'] for row in resultado_final.select('dfu').distinct().collect()]
dfus

In [0]:
import mlflow
mlflow.autolog(disable=True)

In [0]:
teste_pd = resultado_final.toPandas()
teste_pd = teste_pd.rename(columns={'data_cirurgia': 'ds', 'total_cirurgias_realizadas_dia': 'y'})
teste_pd['unique_id'] = teste_pd['dias_antecedencia'].astype('str')

teste_pd

## Separado

# Um modelo diferente para cada dia de antecedência

In [0]:
HORIZON = 50

# Estruturação previsão para o futuro

# Tabela futura

In [0]:
run_id = start_pipeline_log(
    spark=spark,
    pipeline_name="pi_previsao_cirurgias",
    step_name="previsao_eletivas.build_tabela_futura",
    job_id=job_id,
    task_id=task_id,
    notebook_path=notebook_path,
    dt_referencia=timestamp,
    source_table="agendamentos + feriados",
    target_table="resultado_final_df (tabela futura)"
)

try:
    from pyspark.sql import functions as F
    from pyspark.sql.functions import col

    # ==============================================================================
    # 1. PREPARAR BASE DE AGENDAMENTOS
    # ==============================================================================

    agenda_df = agendamentos \
        .where(col('hub') == "Campinas") \
        .where(col('ds_carater').like('%Eletiva%')) \
        .where(~col('ds_agenda').isin(['Ressonância', 'Tomografia'])) \
        .where(col('ds_estabelecimento') == 'Hospital Vera Cruz S/a') \
        .where(col('dt_cancelamento').isNull()) \
        .where(F.to_date('dt_agenda') >= F.current_date()) \
        .where(F.year('dt_agenda') <= 2030) \
        .where(col('ds_status_agenda') != 'Cancelada') \
        .where(col('dt_cancelamento').isNull()) \
        .withColumn('ds', F.to_date('dt_agenda')) \
        .withColumn('dias_antecedencia_fut', (F.datediff(col('ds'), F.current_date())) + 1)

    # Calcular os agendamentos existentes (apenas dias/dfus que têm dados)
    agendamentos_existentes_df = agenda_df.groupBy('ds', 'dfu', 'dias_antecedencia_fut') \
        .agg(F.count('*').alias('agendamentos'))

    # ==============================================================================
    # 2. CRIAR O GABARITO (SCAFFOLD) COMPLETO (DATA x dfu)
    # ==============================================================================

    # Encontrar o intervalo de datas (min e max)
    min_max_datas_df = agenda_df.agg(
        F.max('ds').alias('max_data')
    ).withColumn(
        'min_data', F.current_date()
    )

    # 2.1 Gerar todas as datas
    datas_gabarito_df = min_max_datas_df.select(
        F.explode(
            F.sequence(col('min_data'), col('max_data'), F.expr('interval 1 day'))
        ).alias('ds')
    )

    # 2.2 Gerar os dfus fixos
    # Definimos explicitamente os dfus para garantir que apareçam mesmo sem dados

    lista_dfus = [(dfu,) for dfu in dfus]

    dfus_df = spark.createDataFrame(lista_dfus, ["dfu"])

    # 2.3 Cruzamento (Cross Join) para ter Data x DFU
    # Isso cria uma linha para cada dia E para cada DFU
    gabarito_completo_df = datas_gabarito_df.crossJoin(dfus_df)

    # 2.4 Adicionar a lógica de dias futuros no gabarito
    gabarito_final_df = gabarito_completo_df.withColumn(
        'dias_antecedencia_fut',
        (F.datediff(col('ds'), F.current_date())) + 1
    )

    # ==============================================================================
    # 3. JUNÇÃO E RESULTADO FINAL
    # ==============================================================================

    # Fazer o Left Join do gabarito completo com os agendamentos reais
    resultado_df = gabarito_final_df.join(
        agendamentos_existentes_df,
        ['ds', 'dfu', 'dias_antecedencia_fut'],  # Chave composta inclui o dfu
        'left'
    ).join(
        feriados.withColumnRenamed('data_cirurgia', 'ds'),
        on=['ds'],
        how='left'
    ).withColumn('periodo_baixa', 
        F.when((F.month('ds') == 12) & (F.dayofmonth('ds') >= 22), 1)
        .when((F.month('ds') == 1) & (F.dayofmonth('ds') <= 5), 1)
        .otherwise(0)
    )

    # Preencher os nulos com 0 e ordenar
    resultado_final_df = resultado_df \
        .fillna(0, subset=['agendamentos']) \
        .select('ds', 'dfu', 'dias_antecedencia_fut', 'agendamentos', 'feriado', 'periodo_baixa') \
        .orderBy('ds', 'dfu')

    # Exibir o resultado final
    resultado_final_df.display()

    rows_written = resultado_final_df.count()
    end_pipeline_log(
        spark,
        run_id,
        status="SUCCESS",
        rows_written=rows_written,
    )

except Exception as e:
    safe_error = str(e).replace("'", "''")
    end_pipeline_log(
        spark,
        run_id,
        status="FAILED",
        error_message=safe_error
    )
    raise


In [0]:
tabela_futura = resultado_final_df.where(f'dias_antecedencia_fut <= {HORIZON}').toPandas()
tabela_futura['ds'] = pd.to_datetime(tabela_futura['ds'])
tabela_futura['unique_id'] = tabela_futura['dfu'] + ' / ' + tabela_futura['dias_antecedencia_fut'].astype(str)

tabela_futura

## Previsão futura

## Junto

In [0]:
def cria_feature_estatica(dfu):
    if dfu == 'Ginecologia e Obstetrícia':
        return 1
    elif dfu == 'Outros':
        return 2
    elif dfu == 'Ortopedia':
        return 3
    elif dfu == 'Infantil':
        return 4
    elif dfu == 'Robótica':
        return 5
    elif dfu == 'Cirurgia Geral':
        return 6

In [0]:
run_id = start_pipeline_log(
    spark=spark,
    pipeline_name="pi_previsao_cirurgias",
    step_name="previsao_eletivas.train_modelos_globais",
    job_id=job_id,
    task_id=task_id,
    notebook_path=notebook_path,
    dt_referencia=timestamp,
    source_table="teste_pd + tabela_futura",
    target_table="final_predictions_df (modelos globais por dias_antecedencia)"
)

try:
    # 1. Preparação do DataFrame (fora do loop)
    df = teste_pd.copy() # Usar .copy() para evitar SettingWithCopyWarning
    df['unique_id'] = df['dfu'] + ' / ' + df['unique_id']
    df['feature_dfu'] = df['dfu'].apply(cria_feature_estatica).astype(int)

    df['y'] = df['y'] - df['agendamentos']

    df['ds'] = pd.to_datetime(df['ds'])
    if 'dias_antecedencia' in df.columns:
        df = df.drop(columns=['dias_antecedencia'])

    # 3. Preparar o Loop
    unique_antecedencias = df['unique_id'].str.split(' / ').str[1].unique()
    previsoes = []
    all_models = dict()

    print(f"Iniciando treino de {len(unique_antecedencias)} modelos locais...")

    # 4. Loop para treinar um modelo por unique_id
    for uid in unique_antecedencias:
        print(f"\n--- Treinando modelo para: {uid} ---")

        # 4.1. Filtrar os dados para este ID
        df_id = df[df['unique_id'].str.split(' / ').str[1] == uid].drop(columns=['dfu'])

        # Se um ID não tiver dados de treino suficientes, pule
        if df_id.empty:
            print(f"ID {uid} não possui dados de treino. Pulando.")
            continue

        # 4.2. Instanciar NOVOS modelos e um NOVO MLForecast
        # Isso é crucial para garantir que cada ID tenha seu próprio modelo independente
        all_models[uid] = MLForecast(
            models=[LGBMRegressor(), XGBRegressor(), LinearRegression()],
            freq='D',
            lags=[1, 2, 3, 4, 5, 6, 7],
            # lags=[1, 2, 3, 4, 5, 6, 7, 14],
            lag_transforms={
                1: [RollingMean(window_size=7), RollingMean(window_size=30)],
            },
            date_features=["dayofweek", "month", "year", "week"]
        )

        # 4.3. Treinar o modelo
        all_models[uid].fit(
            df_id, 
            static_features=['feature_dfu'],
            # max_horizon=HORIZON
        )

        # 4.4. Criar o future_dataframe (com features exógenas se houver)
        # O merge com test_id garante que quaisquer colunas extras em 'test'
        # (que não sejam 'y') sejam usadas como X_df (features futuras)
        future_df_id = all_models[uid].make_future_dataframe(HORIZON).merge(
            tabela_futura.drop(columns=['dias_antecedencia_fut', 'dfu']), 
            how='left', 
            on=['ds', 'unique_id']
        ).fillna(0) 

        # 4.5. Fazer a previsão para este ID
        previsao = all_models[uid].predict(
          HORIZON, 
          X_df=future_df_id, 
        )

        # 4.6. Adicionar a previsão à nossa lista
        previsoes.append(previsao)

    # 5. Juntar todas as previsões em um único DataFrame
    if previsoes:
        final_predictions_df = pd.concat(previsoes).reset_index(drop=True)

        print("\n--- ✅ Processo Concluído ---")
        print("DataFrame final de previsões (head):")
        print(final_predictions_df.head())

        print("\nDataFrame final de previsões (tail):")
        print(final_predictions_df.tail())
    else:
        print("Nenhuma previsão foi gerada.")

    rows_read = len(teste_pd)
    rows_written = len(final_predictions_df) if previsoes else 0
    end_pipeline_log(
        spark,
        run_id,
        status="SUCCESS",
        rows_read=rows_read,
        rows_written=rows_written,
    )

except Exception as e:
    safe_error = str(e).replace("'", "''")
    end_pipeline_log(
        spark,
        run_id,
        status="FAILED",
        error_message=safe_error
    )
    raise


In [0]:
final_predictions_tratada = final_predictions_df.copy()

final = final_predictions_df.merge(
    tabela_futura, 
    how='inner', 
    on=['ds', 'unique_id'],
    # on=['ds']
)

# final = final[final['unique_id'].astype(int) == final['dias_antecedencia_fut']]

In [0]:
teste_pd['ds'] = pd.to_datetime(teste_pd['ds'])

In [0]:
for dfu in dfus:
    final[final['unique_id'].str.split(' / ').str[0] == dfu].set_index('ds')[['LGBMRegressor', 'XGBRegressor', 'agendamentos']].plot(figsize=(15, 6), style=['-', '-', '--', '-'])

    plt.title(dfu)
    plt.show();

In [0]:
final_soma = final.copy()
final_soma['LGBMRegressor'] = final_soma['LGBMRegressor'] + final_soma['agendamentos']
final_soma['XGBRegressor'] = final_soma['XGBRegressor'] + final_soma['agendamentos']
final_soma['LinearRegression'] = final_soma['LinearRegression'] + final_soma['agendamentos']

for dfu in dfus:
    final_soma[final_soma['unique_id'].str.split(' / ').str[0] == dfu].set_index('ds')[['LGBMRegressor', 'XGBRegressor', 'agendamentos']].plot(figsize=(15, 6), style=['-', '-', '--', '-'])

    plt.title(f'{dfu} - Global')
    plt.show();

In [0]:
resultados_finais_agendamentos = final_soma.copy()

## Separado

In [0]:
run_id = start_pipeline_log(
    spark=spark,
    pipeline_name="pi_previsao_cirurgias",
    step_name="previsao_eletivas.train_modelos_locais",
    job_id=job_id,
    task_id=task_id,
    notebook_path=notebook_path,
    dt_referencia=timestamp,
    source_table="teste_pd + tabela_futura",
    target_table="final_predictions_all (modelos locais por dfu)"
)

try:
    final_predictions_all = dict()

    for dfu in dfus:
        print(f'Iniciando {dfu}')
        # 1. Preparação do DataFrame (fora do loop)
        df = teste_pd[teste_pd['dfu'] == dfu].drop(columns=['dfu']).copy() # Usar .copy() para evitar SettingWithCopyWarning

        df['y'] = df['y'] - df['agendamentos']

        df['ds'] = pd.to_datetime(df['ds'])
        if 'dias_antecedencia' in df.columns:
            df = df.drop(columns=['dias_antecedencia'])

        # 3. Preparar o Loop
        unique_ids = df['unique_id'].unique()
        previsoes = []
        all_models = dict()

        print(f"Iniciando treino de {len(unique_ids)} modelos locais...")

        # 4. Loop para treinar um modelo por unique_id
        for uid in unique_ids:
            print(f"\n--- Treinando modelo para: {uid} ---")

            # 4.1. Filtrar os dados para este ID
            df_id = df[df['unique_id'] == uid]

            # Se um ID não tiver dados de treino suficientes, pule
            if df_id.empty:
                print(f"ID {uid} não possui dados de treino. Pulando.")
                continue

            # 4.2. Instanciar NOVOS modelos e um NOVO MLForecast
            # Isso é crucial para garantir que cada ID tenha seu próprio modelo independente
            all_models[uid] = MLForecast(
                models=[LGBMRegressor(), XGBRegressor(), LinearRegression()],
                freq='D',
                # lags=[1, 2, 3, 4, 5, 6, 7, 14],
                lags=[1, 2, 3, 4, 5, 6, 7],
                lag_transforms={
                    1: [RollingMean(window_size=7), RollingMean(window_size=30)],
                },
                date_features=["dayofweek", "month", "year", "week"]
            )

            # 4.3. Treinar o modelo local
            all_models[uid].fit(
                df_id, 
                static_features=[],
                # max_horizon=HORIZON
            )

            # 4.4. Criar o future_dataframe (com features exógenas se houver)
            # O merge com test_id garante que quaisquer colunas extras em 'test'
            # (que não sejam 'y') sejam usadas como X_df (features futuras)
            tabela_futura_dfu = tabela_futura[tabela_futura['dfu'] == dfu].drop(columns=['dias_antecedencia_fut', 'unique_id'])

            future_df_id = all_models[uid].make_future_dataframe(HORIZON).merge(
                tabela_futura_dfu, 
                how='left', 
                on=['ds']
            ).fillna(0) 

            # 4.5. Fazer a previsão para este ID
            previsao = all_models[uid].predict(
            HORIZON, 
            X_df=future_df_id, 
            )

            # 4.6. Adicionar a previsão à nossa lista
            previsoes.append(previsao)

        # 5. Juntar todas as previsões em um único DataFrame
        if previsoes:
            final_predictions_df = pd.concat(previsoes).reset_index(drop=True)

            print("\n--- ✅ Processo Concluído ---")
            print("DataFrame final de previsões (head):")
            print(final_predictions_df.head())

            print("\nDataFrame final de previsões (tail):")
            print(final_predictions_df.tail())
        else:
            print("Nenhuma previsão foi gerada.")

        final_predictions_all[dfu] = final_predictions_df

    rows_read = len(teste_pd)
    rows_written = 0
    for _df_dfu in final_predictions_all.values():
        rows_written += len(_df_dfu)
    end_pipeline_log(
        spark,
        run_id,
        status="SUCCESS",
        rows_read=rows_read,
        rows_written=rows_written,
    )

except Exception as e:
    safe_error = str(e).replace("'", "''")
    end_pipeline_log(
        spark,
        run_id,
        status="FAILED",
        error_message=safe_error
    )
    raise


In [0]:
tabela_futura

In [0]:
final = dict()
for dfu in dfus:
    final[dfu] = final_predictions_all[dfu].merge(
        tabela_futura[tabela_futura['dfu'] == dfu].drop(columns=['dfu', 'unique_id']), 
        how='inner', 
        on=['ds']
    )

    final[dfu] = final[dfu][final[dfu]['unique_id'].astype(int) == final[dfu]['dias_antecedencia_fut']]

In [0]:
for dfu in dfus:
    final[dfu].set_index('ds')[['LGBMRegressor', 'XGBRegressor', 'agendamentos']].plot(figsize=(15, 6), style=['-', '-', '--'])

    plt.title(dfu)
    plt.show();

In [0]:
final_soma = dict()

for dfu in final.keys():
    final_soma[dfu] = final[dfu].copy()

    final_soma[dfu]['LGBMRegressor'] = final_soma[dfu]['LGBMRegressor'] + final_soma[dfu]['agendamentos']
    final_soma[dfu]['XGBRegressor'] = final_soma[dfu]['XGBRegressor'] + final_soma[dfu]['agendamentos']
    final_soma[dfu]['LinearRegression'] = final_soma[dfu]['LinearRegression'] + final_soma[dfu]['agendamentos']

for dfu in dfus:
    final_soma[dfu].set_index('ds')[['LGBMRegressor', 'XGBRegressor', 'agendamentos']].plot(figsize=(15, 6), style=['-', '-', '--'])

    plt.title(f'{dfu} - Local')
    plt.show();

In [0]:
resultados_finais_agendamentos_local = None

for dfu in final_soma.keys():
    dataframe = final_soma[dfu].copy()

    dataframe['unique_id'] = dfu + ' / ' + dataframe['unique_id']
    dataframe['dfu'] = dfu

    if resultados_finais_agendamentos_local is None:
        resultados_finais_agendamentos_local = dataframe
    else:
        resultados_finais_agendamentos_local = pd.concat([resultados_finais_agendamentos_local, dataframe], ignore_index=True)

In [0]:
resultados_finais_agendamentos_local

In [0]:
for dfu in dfus:
    plot_df = teste_pd[teste_pd['dfu'] == dfu]
    plot_df[(plot_df['unique_id'] == '1') & (plot_df['ds'] >= pd.to_datetime('2024-11-25')) & (plot_df['ds'] <= pd.to_datetime('2025-01-13'))].set_index('ds')['y'].plot(figsize=(15, 6))
    plt.title(dfu)
    plt.show();

In [0]:
teste_pd


# Previsão além dos agendamentos

In [0]:
LONG_HORIZON = 365

In [0]:
df = teste_pd
df = df[df['unique_id'] == '1'].drop(columns=['unique_id']).rename(columns={'dfu': 'unique_id'})

df['ds'] = pd.to_datetime(df['ds'])
df = df.drop(columns=['dias_antecedencia', 'agendamentos'])

# Conversão de data e limpeza
df['ds'] = pd.to_datetime(df['ds'])
cols_to_drop = ['dias_antecedencia', 'agendamentos']
df = df.drop(columns=[c for c in cols_to_drop if c in df.columns])

# Lista para armazenar os resultados de cada modelo
lista_previsoes = []

# Obtemos a lista de todos os DFUs únicos
dfus_unicos = df['unique_id'].unique()

for dfu_id in dfus_unicos:
    df_subset = df[df['unique_id'] == dfu_id].copy()

    test_subset = df_subset.sort_values(by=['ds']).tail(LONG_HORIZON)
    train_subset = df_subset.drop(test_subset.index).sort_values(by=['ds'])

    fcst = MLForecast(
        models=[LGBMRegressor(verbosity=-1), XGBRegressor(), LinearRegression()], 
        freq='D',
        lags=[1, 2, 3, 4, 5, 6, 7, 14],
        lag_transforms={
            1: [RollingMean(window_size=7), RollingMean(window_size=30)],
        },
        date_features=["dayofweek", "month", "year", "week", "quarter", "is_month_start", "is_month_end"]
    )

    # Treina o modelo apenas com os dados deste DFU
    fcst.fit(train_subset, static_features=[])

    # Previsão
    future = fcst.predict(LONG_HORIZON, X_df=test_subset.drop(columns=['y']))
    
    # Adicionamos os valores reais para comparação futura
    future = future.merge(test_subset[['unique_id', 'ds', 'y']], on=['unique_id', 'ds'], how='left')
    
    # Guarda o resultado na lista
    lista_previsoes.append(future)

# 3. Junção dos resultados
if lista_previsoes:
    df_resultado_final = pd.concat(lista_previsoes, ignore_index=True)
    print("Processo finalizado.")
else:
    print("Nenhuma previsão gerada.")
    df_resultado_final = pd.DataFrame()

# Visualização das primeiras linhas
display(df_resultado_final.head())

In [0]:
for uid in df_resultado_final['unique_id'].unique():
    plt.figure(figsize=(15, 6))
    plt.plot(df_resultado_final[df_resultado_final['unique_id'] == uid]['ds'], df_resultado_final[df_resultado_final['unique_id'] == uid]['LGBMRegressor'].round(0), label='LGBMRegressor')
    plt.plot(df[df['unique_id'] == uid].sort_values('ds')['ds'][-LONG_HORIZON:], df[df['unique_id'] == uid].sort_values('ds')['y'][-LONG_HORIZON:], label='y')
    plt.title(f'Unique ID: {uid}')
    plt.legend()
    plt.show()

In [0]:
erros_locais = evaluate(df_resultado_final, metrics=[mae])

In [0]:
erros_locais

In [0]:
future


## Previsão

In [0]:
tabela_futura = resultado_final_df.where(f'dias_antecedencia_fut <= {HORIZON}').toPandas()
tabela_futura['ds'] = pd.to_datetime(tabela_futura['ds'])
tabela_futura['unique_id'] = tabela_futura['dfu'] + ' / ' + tabela_futura['dias_antecedencia_fut'].astype(str)

tabela_futura

In [0]:
tabela_futura

In [0]:
feriados_pd = feriados.withColumnRenamed('data_cirurgia', 'ds').toPandas()
feriados_pd['ds'] = pd.to_datetime(feriados_pd['ds'])

In [0]:
run_id = start_pipeline_log(
    spark=spark,
    pipeline_name="pi_previsao_cirurgias",
    step_name="previsao_eletivas.train_previsao_longo_prazo",
    job_id=job_id,
    task_id=task_id,
    notebook_path=notebook_path,
    dt_referencia=timestamp,
    source_table="teste_pd + feriados",
    target_table="previsoes_alem (previsão LONG_HORIZON sem agendamentos)"
)

try:
    # Target
    df = teste_pd
    df = df[df['unique_id'] == '1'].drop(columns=['unique_id']).rename(columns={'dfu': 'unique_id'})

    df['ds'] = pd.to_datetime(df['ds'])
    df = df.drop(columns=['dias_antecedencia', 'agendamentos'])

    # Features
    feriados_pd = feriados\
        .withColumnRenamed('data_cirurgia', 'ds')\
        .withColumn('periodo_baixa', 
            F.when((F.month('ds') == 12) & (F.dayofmonth('ds') >= 22), 1)
            .when((F.month('ds') == 1) & (F.dayofmonth('ds') <= 5), 1)
            .otherwise(0)
        ).toPandas()
    feriados_pd['ds'] = pd.to_datetime(feriados_pd['ds'])

    # Conversão de data e limpeza
    df['ds'] = pd.to_datetime(df['ds'])
    cols_to_drop = ['dias_antecedencia', 'agendamentos']
    df = df.drop(columns=[c for c in cols_to_drop if c in df.columns])

    # Lista para armazenar os resultados de cada modelo
    lista_previsoes = []

    # Obtemos a lista de todos os DFUs únicos
    dfus_unicos = df['unique_id'].unique()

    for dfu_id in dfus_unicos:
        df_subset = df[df['unique_id'] == dfu_id].sort_values('ds').copy()

        fcst = MLForecast(
            models=[LGBMRegressor(verbosity=-1), XGBRegressor(), LinearRegression()], 
            freq='D',
            lags=[1, 2, 3, 4, 5, 6, 7, 14],
            lag_transforms={
                1: [RollingMean(window_size=7), RollingMean(window_size=30)],
            },
            date_features=["dayofweek", "month", "year", "week", "quarter", "is_month_start", "is_month_end"]
        )

        # Treina o modelo apenas com os dados deste DFU
        fcst.fit(df_subset, static_features=[])

        future_df_id = fcst.make_future_dataframe(LONG_HORIZON).merge(
            feriados_pd, 
            how='left', 
            on=['ds']
        )

        # Previsão
        future = fcst.predict(LONG_HORIZON, X_df=future_df_id)

        # Guarda o resultado na lista
        lista_previsoes.append(future)

    # 3. Junção dos resultados
    if lista_previsoes:
        previsoes_alem = pd.concat(lista_previsoes, ignore_index=True)
        print("Processo finalizado.")
    else:
        print("Nenhuma previsão gerada.")
        previsoes_alem = pd.DataFrame()

    # Visualização das primeiras linhas
    display(previsoes_alem.head())

    rows_read = len(teste_pd)
    rows_written = len(previsoes_alem)
    end_pipeline_log(
        spark,
        run_id,
        status="SUCCESS",
        rows_read=rows_read,
        rows_written=rows_written,
    )

except Exception as e:
    safe_error = str(e).replace("'", "''")
    end_pipeline_log(
        spark,
        run_id,
        status="FAILED",
        error_message=safe_error
    )
    raise


In [0]:
previsoes_alem['previsao_alem'] = previsoes_alem['LGBMRegressor']
previsoes_alem['origem'] = "previsao_sem_agendamento"
previsoes_alem['dfu'] = previsoes_alem['unique_id']

previsoes_alem


# Resultado Final

In [0]:
resultado_final_backup = resultados_finais_agendamentos.copy()

In [0]:
run_id = start_pipeline_log(
    spark=spark,
    pipeline_name="pi_previsao_cirurgias",
    step_name="previsao_eletivas.combine_global_local",
    job_id=job_id,
    task_id=task_id,
    notebook_path=notebook_path,
    dt_referencia=timestamp,
    source_table="resultados_finais_agendamentos + resultados_finais_agendamentos_local",
    target_table="res_finais"
)

try:
    import numpy as np

    resultados_finais_agendamentos['previsao_global'] = resultados_finais_agendamentos['LGBMRegressor']
    resultados_finais_agendamentos['origem'] = 'agendamento'

    resultados_finais_agendamentos_local['previsao_local'] = resultados_finais_agendamentos_local['LGBMRegressor']
    resultados_finais_agendamentos_local['origem'] = 'agendamento'

    res_finais = resultados_finais_agendamentos.merge(
        resultados_finais_agendamentos_local[['dfu', 'ds', 'previsao_local']],
        on=['dfu', 'ds'],
        how='left'
    )

    res_finais['previsao'] = np.max([res_finais['previsao_global'], res_finais['previsao_local']], axis=0)

    res_finais

    rows_written = len(res_finais)
    end_pipeline_log(
        spark,
        run_id,
        status="SUCCESS",
        rows_written=rows_written,
    )

except Exception as e:
    safe_error = str(e).replace("'", "''")
    end_pipeline_log(
        spark,
        run_id,
        status="FAILED",
        error_message=safe_error
    )
    raise


In [0]:
plot_df['ds'][-HORIZON*2:]

In [0]:
previsoes_alem

In [0]:
df_historico = teste_pd[teste_pd['unique_id'] == '1']
df_historico['origem'] = 'histórico'

ds_previsoes_agendamentos = res_finais['ds'].unique()
previsoes_alem_filtrada = previsoes_alem[~previsoes_alem['ds'].isin(ds_previsoes_agendamentos)]

df_historico_previsao = pd.concat([df_historico[['ds', 'dfu', 'y', 'origem']].sort_values('ds'), res_finais[['ds', 'dfu', 'previsao', 'origem']], previsoes_alem_filtrada[['ds', 'dfu', 'previsao_alem', 'origem']]]).sort_values('ds')

for dfu in dfus:
    plot_df = df_historico_previsao[df_historico_previsao['dfu'] == dfu]
    
    plt.figure(figsize=(15, 6))
    plt.plot(plot_df['ds'][-(LONG_HORIZON+HORIZON*6):], plot_df['y'][-(LONG_HORIZON+HORIZON*6):], label='y')
    plt.plot(plot_df['ds'], plot_df['previsao'].round(0), label='previsao_agendamento')
    plt.plot(plot_df['ds'], plot_df['previsao_alem'].round(0), label='previsao_alem')
    plt.title(f'DFU: {dfu}')
    plt.legend()
    plt.show()

# Utilizando previsão de agendamentos para treinar a previsão longa

In [0]:
df_treino_longo = teste_pd[teste_pd['unique_id'] == '1'][['ds', 'dfu', 'y', 'feriado', 'periodo_baixa']]
df_treino_longo = pd.concat(
    [df_treino_longo, res_finais.rename(columns={'previsao': 'y'})[['ds', 'dfu', 'y', 'feriado', 'periodo_baixa']]]
).sort_values('ds')

df_treino_longo

In [0]:
# Target
df = df_treino_longo.copy()
df = df.rename(columns={'dfu': 'unique_id'})

# Features
feriados_pd = feriados\
    .withColumnRenamed('data_cirurgia', 'ds')\
    .withColumn('periodo_baixa', 
        F.when((F.month('ds') == 12) & (F.dayofmonth('ds') >= 22), 1)
        .when((F.month('ds') == 1) & (F.dayofmonth('ds') <= 5), 1)
        .otherwise(0)
    ).toPandas()
feriados_pd['ds'] = pd.to_datetime(feriados_pd['ds'])

# Conversão de data e limpeza
df['ds'] = pd.to_datetime(df['ds'])

# Lista para armazenar os resultados de cada modelo
lista_previsoes = []

# Obtemos a lista de todos os DFUs únicos
dfus_unicos = df['unique_id'].unique()

for dfu_id in dfus_unicos:
    df_subset = df[df['unique_id'] == dfu_id].sort_values('ds').copy()

    fcst = MLForecast(
        models=[LGBMRegressor(verbosity=-1), XGBRegressor(), LinearRegression()], 
        freq='D',
        lags=[1, 2, 3, 4, 5, 6, 7, 14],
        lag_transforms={
            1: [RollingMean(window_size=7), RollingMean(window_size=30)],
        },
        date_features=["dayofweek", "month", "year", "week", "quarter", "is_month_start", "is_month_end"]
    )

    # Treina o modelo apenas com os dados deste DFU
    fcst.fit(df_subset, static_features=[])

    future_df_id = fcst.make_future_dataframe(LONG_HORIZON - HORIZON).merge(
        feriados_pd, 
        how='left', 
        on=['ds']
    )

    # Previsão
    future = fcst.predict(LONG_HORIZON - HORIZON, X_df=future_df_id)
    
    # Guarda o resultado na lista
    lista_previsoes.append(future)

# 3. Junção dos resultados
if lista_previsoes:
    previsoes_alem_agend = pd.concat(lista_previsoes, ignore_index=True)
    print("Processo finalizado.")
else:
    print("Nenhuma previsão gerada.")
    previsoes_alem_agend = pd.DataFrame()

# Visualização das primeiras linhas
display(previsoes_alem_agend.head())

In [0]:
previsoes_alem_agend['previsao_alem'] = previsoes_alem_agend['LGBMRegressor']
previsoes_alem_agend['origem'] = "previsao_sem_agendamento"
previsoes_alem_agend['dfu'] = previsoes_alem_agend['unique_id']

df_historico = teste_pd[teste_pd['unique_id'] == '1']
df_historico['origem'] = 'histórico'

ds_previsoes_agendamentos = res_finais['ds'].unique()
previsoes_alem_filtrada = previsoes_alem_agend

df_historico_previsao_agend = pd.concat([df_historico[['ds', 'dfu', 'y', 'origem']].sort_values('ds'), res_finais[['ds', 'dfu', 'previsao', 'origem']], previsoes_alem_filtrada[['ds', 'dfu', 'previsao_alem', 'origem']]]).sort_values('ds')

for dfu in dfus:
    plot_df = df_historico_previsao_agend[df_historico_previsao_agend['dfu'] == dfu]
    
    plt.figure(figsize=(15, 6))
    plt.plot(plot_df['ds'][-(LONG_HORIZON+HORIZON*6):], plot_df['y'][-(LONG_HORIZON+HORIZON*6):], label='y')
    plt.plot(plot_df['ds'], plot_df['previsao'].round(0), label='previsao_agendamento')
    plt.plot(plot_df['ds'], plot_df['previsao_alem'].round(0), label='previsao_alem')
    plt.title(f'DFU: {dfu} - Usa Prev. Agendamento')
    plt.legend()
    plt.show()


# Pós-processamento

In [0]:
tabela_futura_pos = resultado_final_df.toPandas()
tabela_futura_pos['ds'] = pd.to_datetime(tabela_futura_pos['ds'])
tabela_futura_pos[['ds', 'dfu', 'agendamentos']]

tabela_futura_pos

In [0]:
run_id = start_pipeline_log(
    spark=spark,
    pipeline_name="pi_previsao_cirurgias",
    step_name="previsao_eletivas.post_processing",
    job_id=job_id,
    task_id=task_id,
    notebook_path=notebook_path,
    dt_referencia=timestamp,
    source_table="df_historico_previsao + tabela_futura_pos",
    target_table="resultado (previsão final)"
)

try:
    resultado = df_historico_previsao.dropna(subset=['previsao', 'previsao_alem'], how='all')[['ds', 'dfu', 'previsao', 'previsao_alem']]

    resultado['previsao_final'] = resultado['previsao'].combine_first(resultado['previsao_alem'])

    # Adicionando agendamentos
    resultado = resultado.merge(tabela_futura_pos[['ds', 'dfu', 'agendamentos']], on=['ds', 'dfu'], how='left')
    resultado['agendamentos'] = resultado['agendamentos'].fillna(0)

    # Não permitindo que a previsão seja menor que o agendamento
    resultado['previsao_final'] = resultado[['previsao_final', 'agendamentos']].max(axis=1)

    # Arredondando
    resultado['previsao_final'] = resultado['previsao_final'].round(0).astype(int)

    # Limitando em 0
    resultado['previsao_final'] = resultado['previsao_final'].clip(lower=0)

    # Excluindo colunas extras
    resultado = resultado[['ds', 'dfu', 'agendamentos','previsao_final']]

    # Adicionando data_processamento
    resultado['data_processamento'] = today_date

    # Adicionando distancia_previsao como número inteiro de dias
    resultado['distancia_previsao'] = (resultado['ds'].dt.date - today_date).apply(lambda x: x.days + 1).astype(int)

    resultado

    rows_read = len(df_historico_previsao)
    rows_written = len(resultado)
    end_pipeline_log(
        spark,
        run_id,
        status="SUCCESS",
        rows_read=rows_read,
        rows_written=rows_written,
    )

except Exception as e:
    safe_error = str(e).replace("'", "''")
    end_pipeline_log(
        spark,
        run_id,
        status="FAILED",
        error_message=safe_error
    )
    raise


In [0]:
resultado_spark = spark.createDataFrame(resultado).withColumn('import_date', F.current_timestamp())
resultado_spark.display()

# Salvando

In [0]:
from pathlib import Path
import sys
import importlib
import os
import unidecode

current_dir = os.getcwd()
parent_dir = os.path.dirname(current_dir)
sys.path.append(parent_dir)

import utils
importlib.reload(utils)
from utils import write_to_table

In [0]:
table_name = f"{catalog}.ds_pipeline.previsao_eletivas"
 
resultado_spark = apply_schema_from_table(
    resultado_spark,
    table_name,
    spark=spark
)

In [0]:
run_id = start_pipeline_log(
    spark=spark,
    pipeline_name="pi_previsao_cirurgias",
    step_name="previsao_eletivas.save_previsao_eletivas",
    job_id=job_id,
    task_id=task_id,
    notebook_path=notebook_path,
    dt_referencia=timestamp,
    source_table="resultado_spark",
    target_table="{catalog}.ds_pipeline.previsao_eletivas"
)

try:
    table_name = f"{catalog}.ds_pipeline.previsao_eletivas"
    write_to_table(
        df=resultado_spark,
        table_name=table_name,
        data_column="data_processamento",
        today_date=today_date
    )

    # Display recent data
    display(
        spark.table(table_name)
             .where(F.col("data_processamento") == F.to_date(F.lit(today_date)))
             .orderBy(F.desc("data_processamento"), "dfu", "ds")
             .limit(1000)
    )

    rows_read = resultado_spark.count()
    rows_written = resultado_spark.count()
    end_pipeline_log(
        spark,
        run_id,
        status="SUCCESS",
        rows_read=rows_read,
        rows_written=rows_written,
    )

except Exception as e:
    safe_error = str(e).replace("'", "''")
    end_pipeline_log(
        spark,
        run_id,
        status="FAILED",
        error_message=safe_error
    )
    raise


# Visualização

In [0]:
import seaborn as sns

sns.set(rc={'figure.figsize':(15, 6)})
g = sns.FacetGrid(resultado, col='dfu', col_wrap=1, height=6, aspect=2)
g.map(sns.lineplot, 'ds', 'previsao_final')